# 05 — Election-only spenders vs ongoing advocates

**The headline analytical question**: within Australian political-adjacent Facebook advertising, can we empirically separate *election-only* spenders (groups that exist primarily to advertise around an election) from *ongoing advocates* (NGOs and issue groups that run ads continuously)?

**Why this question matters**: it sounds simple but it gets at the structure of political-adjacent ad spending — how much of what looks like political advertising is *actually* tied to the electoral cycle, versus a constant background of issue advocacy that just happens to be louder around elections?

**Method outline**:

1. **Filter to the political-adjacent corpus** using the classification and topic-labelling work from notebooks 02–03:
   - Keep `match_type ∈ {candidate, party_org}` — registered party advertising.
   - Keep `match_type IS NULL` **only when** the LDA-derived topic label is non-noise — the residual political-adjacent advertisers (NGOs, advocacy, issue campaigns).
   - Drop `match_type = 'government'` — government departments are out of scope.
   - Drop residual ads whose topic label is `noise_*` — commercial/streaming/junk that LDA surfaced.
2. **Split each advertiser's ad activity** into pre-election (Nov 2021 – May 2022) and post-election (May 2022 – Nov 2022) halves.
3. **Compute a persistence ratio** per advertiser: `post_spend / pre_spend`. Election-only advertisers have ratios near zero; ongoing advocates have ratios near 1.
4. **Visualise** the two groups: who dominates each, what topics they push, how their temporal profiles differ.

Window: 6 months before to 6 months after the 21 May 2022 federal election (set in [01_data_loading.ipynb](01_data_loading.ipynb)).

## 1. Setup — load v3 and filter to the political-adjacent corpus

v3 parquet has `political_party`, `match_type`, `topic_id`, `topic_label`, and `category` columns attached. We need ads that are *political* (by classification or topic) but *non-government* and *non-noise*.

Build a small `LABELS` Spark DataFrame from `topic_labels.csv` for the join, then derive a boolean `is_political` flag:

- `match_type ∈ {candidate, party_org}` → political (party/candidate-driven)
- `match_type IS NULL AND category ≠ 'noise'` → political-adjacent advocacy (LDA-passed)
- otherwise → drop

Cache the filtered corpus — every subsequent section scans it.

## 2. Persistence ratio per advertiser

**Metric**: `persistence = post_spend / pre_spend` per `bylines` (the authoriser identity).

- Pre = ads created Nov 2021 – May 2022
- Post = ads created May 2022 – Nov 2022
- Filter to advertisers with `pre_spend ≥ $X` (e.g. $1k) so we're not analysing one-off advertisers

**Bands**:

- `persistence < 0.2` → **election-only** (almost no post-election activity)
- `0.2 ≤ persistence ≤ 0.8` → mixed / transitional
- `persistence > 0.8` → **ongoing advocate** (post-election spending similar to pre)

Output: one row per byline with `(total_spend, pre_spend, post_spend, persistence, group)`.

## 3. The two groups — who's in each

**Top election-only advertisers** (sorted by total spend, then persistence ascending):

Expected: UAP (Streamotion-style massive spend, concentrated in 4–6 weeks), party central offices (ALP, LP, AJP, LNP), candidate pages, Climate 200's federal independents campaign, attack-funder organisations (Advance Australia, Solutions for Australia).

**Top ongoing advocates** (sorted by total spend, persistence descending):

Expected: environmental NGOs (Greenpeace, ACF, Wilderness Society), humanitarian NGOs (Amnesty, Oxfam, Save the Children, UNHCR), industry/civic bodies (AAA, Pharmacy Guild, ABC Friends), advocacy unions (UWU, AEU).

**Headline chart**: scatter of `(total_spend, persistence)` per top-N byline, coloured by `match_type`, log-scaled on spend. Quadrants emerge naturally — election-only top-left, ongoing top-right.

## 4. Temporal profile — election-only vs ongoing

**Question**: how different do the two groups' weekly spending curves look?

**Approach**: weekly spend, stacked area or two-line chart with election-only and ongoing as separate series. Election-day vertical marker. Expect:

- Election-only: sharp pre-election spike, near-zero post-election
- Ongoing: roughly flat across the 12-month window, with possible election-period bump

If the shapes are dramatically different, the bimodal claim is empirically demonstrated by curves alone.

## 5. Topic composition — what each group cares about

**Question**: do election-only and ongoing advertisers push different topics?

**Approach**: cross-tab `group × category` (or `group × topic_label` for finer detail), normalised to row percentages. Heatmap.

**Expected pattern**:

- **Election-only** dominated by: `political_advocacy` (Climate 200, anti-Labor), `election_generic`, `anti_morrison_climate` (if topic survives).
- **Ongoing** dominated by: `humanitarian_rights` (Amnesty, Oxfam), `climate` (Greenpeace conservation, Wilderness, ocean campaigns), `cost_of_living_medicines` (Pharmacy Guild).

Demonstrates the *structural difference* between the two groups: they're not just spending at different times, they're advocating for different things.

## 6. Discussion and main message

**Three findings**:

1. **The split is real and quantifiable.** Persistence ratio cleanly separates the two groups — election-only advertisers (parties, candidates, election-funders) have post/pre spend ratios <0.2; ongoing advocates (NGOs, civic organisations) have ratios >0.8. Mixed advertisers in the middle band are a small minority.
2. **The temporal profiles look completely different.** Election-only spending peaks 10×–50× above its post-election baseline; ongoing spending varies <2× across the same window.
3. **Each group pushes structurally different topics.** Election-only spending concentrates on partisan and election-mechanics topics (vote, election, anti-X framing). Ongoing spending concentrates on issue advocacy (climate, refugees, women's safety, medicines costs).

**Main message for non-technical stakeholders**: Australian political Facebook advertising in 2022 was structurally two separate ecosystems — a transient election-only layer that disappears once the polls close, and a continuous advocacy layer that runs year-round. Lumping them together as "political advertising" obscures who's actually doing what. For example, an analyst studying "political ad spend during the 2022 election" needs to know roughly half the dollars came from election-only PAC-style advertisers and the other half from ongoing NGO advocacy that would have been spending anyway.

## Future work (out of scope for this report)

- **Sentiment overlay**: does election-only advertising skew more negative (attack ads) than ongoing advocacy (donation/issue appeals)? Notebook 04 has the design; deferred for word-count reasons.
- **Demographic targeting differences**: does the election-only group target different age/gender segments than ongoing advocates? Requires `demographic_distribution` array parsing.
- **Geographic spread**: are election-only advertisers more state-concentrated (marginal-seat targeting)? Requires `delivery_by_region` array parsing.
- **Voice referendum comparison**: extending to the 2023 referendum would test whether the same election-only/ongoing structure recurs — but the Aug 2023 FB API breakage makes that infeasible with this dataset.

## Status

Stub only — implementation pending. Each section above gets one or two code cells. The headline chart (section 3's persistence scatter) is the figure the report leads with.